In [ ]:
!pip install -q cassio datasets langchain openai tiktoken

In [ ]:
!pip install -U langchain-community langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Cassandra
#from langchain.indexes import VectorstoreIndexCreator
from langchain_community.document_loaders import TextLoader

In [ ]:
import cassio

In [ ]:
 !pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.0 MB/s eta 0:00:00


In [ ]:
from PyPDF2 import PdfReader

In [ ]:
ASTRA_DB_APPLICATION_TOKEN = "AstraCS:vOCsXnobhPDvkgHHIvpLnDOH:926c09a095ef845770fecf2c2fcd62de2052a70ed97020c4a3e94bef799fc280"
ASTRA_DB_ID = "d86997ff-85d5-43ad-a2a5-63f654f76aef" # enter database ID

OPEN_API_KEY = ""

In [ ]:
PdfReader = PdfReader('hema.pdf')

In [ ]:
from typing_extensions import Concatenate

#read text from pdf
raw_text = ''
for i, page in enumerate(PdfReader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [ ]:
raw_text

'Hemasree\n \nUppaluri\n \nVenkata\n \nSyamala\n \n                                       \nuvs.hema@gmail.com\n \n|\n \n602-813-8372\n \n|\n \nLinkedIn\n \n \nTECHNICAL\n \nSKILLS\n \nProduct\n \nOperations\n \n&\n \nDelivery:\n \nJira\n \n(Bug\n \nTriage/Backlog\n \nManagement),\n \nConfluence\n \n(Documentation/Release\n \nNotes),\n \nAgile/Scrum,\n \nSoftware\n \nDevelopment\n \nLife\n \nCycle\n \n(SDLC),\n \nUser\n \nAcceptance\n \nTesting\n \n(UAT),\n \nRelease\n \nReadiness.\n \nQuality\n \nAssurance:\n \nManual\n \nQA\n \nTesting,\n \nSmoke\n \n&\n \nRegression\n \nTesting,\n \nDefect\n \nValidation,\n \nEnvironment\n \nDetail\n \nCapturing,\n \nTest\n \nCase\n \nDevelopment.\n \nProgramming\n \n&\n \nFrameworks:\n \nPython\n \n(Automation/ETL),\n \nSQL\n \n(PostgreSQL,\n \nMySQL,\n \nMS\n \nSQL\n \nServer,\n \nOracle),\n \nHTML,\n \nCSS,\n \nFastAPI,\n \nPydantic\n \nData\n \n&\n \nAI\n \nPractices:\n \nGoogle\n \nBigQuery,\n \nGoogle\n \nCloud\n \nFunctions,\n \nNLP,\n \nSent

Intialise the connection to your database

In [ ]:
cassio.init(token = ASTRA_DB_APPLICATION_TOKEN , database_id = ASTRA_DB_ID)

Create the langchain embedding and LLM objects for later usage:

In [ ]:
llm = OpenAI(openai_api_key=OPEN_API_KEY)
embedding = OpenAIEmbeddings(openai_api_key=OPEN_API_KEY)

Create your langchain vector store , backed by Astra DB!!

In [ ]:
astra_vector_store = Cassandra (
    embedding = embedding,
    table_name = "langchain_test",
    keyspace = None,
    session = cassio.session,
)

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
#we need to split the text using character text split such that it should not increase token size

text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len,
)

texts = text_splitter.split_text(raw_text)


In [ ]:
texts[:50]

Load the dataset into vector store

In [ ]:
astra_vector_store.add_texts(texts[:50])
print("Inserted %i headlines :" len(texts[:50]))
astra_vector_index = VectorstoreIndexWrapper(
    vectorstore = astra_vector_store
)
